# Modelado Titanic — cuaderno 04

Diseño, entrenamiento y evaluación de modelos. Correr de arriba abajo. El conjunto de prueba (`X_test`) se abre **una sola vez** (Paso 62). Las decisiones congeladas viven en `flows/05_ml_models.py`; los informes en `reports/problema_ml.md`.

In [ ]:
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ml = importlib.import_module('flows.05_ml_models')
from flows.common.paths import BRONZE, MODELS, BRONZE_UNLABELED_NAME

sns.set_theme(style='whitegrid', context='notebook')
X, y = ml.load_dataset()
print(X.shape, y.mean())

## 10. Formulación (Paso 39)

| Decisión | Respuesta |
|---|---|
| Tipo de problema | Clasificación binaria supervisada |
| Variable objetivo | `Survived` (1 = sobrevivió) |
| Unidad y momento | Un pasajero, *al embarcar* |
| Métrica principal | ROC-AUC: no depende del umbral y con 38% de positivos es estable |
| Métricas secundarias | precisión, recall, exactitud, F1 |
| Costo de errores | para la aseguradora, un falso 'sobrevive' sobreestima el riesgo aceptado; un falso 'no sobrevive' subestima exposición |
| Criterio de éxito | superar la regla de género (~0.79 de exactitud) y el ROC-AUC del dummy (0.5) |

### Paso 40. Auditoría de variables y fuga de datos

| Columna | ¿Existe antes del evento? | ¿Riesgo de fuga? | Decisión |
|---|---|---|---|
| `Survived`, `Survivor` | No (objetivo) | Fuga directa | Excluir |
| `PassengerId`, `Ticket`, `Name` | Sí | Identificadores | Excluir (título sí, vía `TTitulo`) |
| `NSex`, `NEmbarked`, `NCabin`, `NTitulo` | — | Duplican con orden arbitrario | Excluir (usar one-hot) |
| `Age` imputada | — | Estadística de todos los datos | Usar `AgeOriginal` |
| `Cabin`, `TCabin`, `HasCabin` | Dudoso | Posible artefacto del registro | Evaluar con y sin (Paso 61) |
| `Person` | Derivada | Redundante | Excluir del modelo base |

### Paso 41. División estratificada (el test se congela desde aquí)

In [ ]:
X_train, X_test, y_train, y_test = ml.split(X, y)
print(len(X_train), 'entrenamiento /', len(X_test), 'prueba')
print('tasa train:', y_train.mean().round(3), '| tasa test:', y_test.mean().round(3))

### Paso 42. Baselines

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    ml.settings.cv_folds, shuffle=True, random_state=ml.settings.random_state)

dummy = ml.build_pipeline(ml.get_candidates()['dummy'])
res_dummy = cross_validate(dummy, X_train, y_train, cv=cv,
                           scoring=['accuracy', 'f1', 'roc_auc'])
print('dummy (CV):', {k: round(res_dummy[f'test_{k}'].mean(), 3)
                      for k in ['accuracy', 'f1', 'roc_auc']})

pred_genero = (X_train['Sex'] == 'female').astype(int).to_numpy()
print('regla de género:', {
    'accuracy': round(accuracy_score(y_train, pred_genero), 3),
    'precision': round(precision_score(y_train, pred_genero), 3),
    'recall': round(recall_score(y_train, pred_genero), 3),
    'f1': round(f1_score(y_train, pred_genero), 3)})
# ¡Esa es la vara que hay que superar!

### Paso 43. Demostración de la fuga de datos

In [ ]:
print('edad media mujeres TODOS los datos:', X.loc[X['Sex'] == 'female', 'Age'].mean().round(2))
print('edad media mujeres SOLO entrenamiento:',
      X_train.loc[X_train['Sex'] == 'female', 'Age'].mean().round(2))
# La diferencia es pequeña aquí, pero el método es el error: en datasets con más faltantes
# o grupos pequeños, estadísticas calculadas con TODOS los datos filtran información
# del conjunto de prueba hacia el entrenamiento.

### Pasos 44-46. Verificar el preprocesamiento

In [ ]:
pipe_demo = ml.build_pipeline(ml.get_candidates()['dummy'])
pre = pipe_demo[:-1].fit(X_train)
Z = pre.transform(X_test)
print('¿algún NaN?:', np.isnan(Z).any(), '| columnas:', Z.shape[1])
print(list(pre['prep'].get_feature_names_out())[:10])

## 11. Validación cruzada de candidatos (Pasos 47-49)

In [ ]:
tabla = ml.cross_validate_candidates(X_train, y_train)
tabla

In [ ]:
filas = []
for nombre, modelo in ml.get_candidates().items():
    res = cross_validate(ml.build_pipeline(modelo), X_train, y_train, cv=cv,
                         scoring=[ml.PRIMARY_METRIC])
    filas += [{'model': nombre, 'roc_auc': s} for s in res[f'test_{ml.PRIMARY_METRIC}']]
pliegues = pd.DataFrame(filas)
sns.boxplot(data=pliegues, x='model', y='roc_auc')
plt.title('ROC-AUC por pliegue')
plt.show()
tabla[['model', 'train_roc_auc_mean', 'test_roc_auc_mean', 'gap_train_test_auc']]

## 12. Evaluación en profundidad (Pasos 51-55)

Predicciones fuera de muestra: cada pasajero se predice con un modelo que no lo vio.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

nombre_finalista = tabla.iloc[0]['model']
print('finalista provisional:', nombre_finalista)
pipe_finalista = ml.build_pipeline(ml.get_candidates()[nombre_finalista])
proba = cross_val_predict(pipe_finalista, X_train, y_train, cv=cv, method='predict_proba')[:, 1]
pred = (proba >= 0.5).astype(int)
ConfusionMatrixDisplay.from_predictions(y_train, pred)
plt.show()
print(classification_report(y_train, pred))

In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
RocCurveDisplay.from_predictions(y_train, proba, ax=axes[0])
PrecisionRecallDisplay.from_predictions(y_train, proba, ax=axes[1])
axes[0].set_title('ROC'); axes[1].set_title('Precisión-Recall')
plt.tight_layout(); plt.show()

### Paso 53. Calibración y umbral

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, precision_recall_curve

real, estimada = calibration_curve(y_train, proba, n_bins=8)
plt.plot(estimada, real, marker='o', label='modelo')
plt.plot([0, 1], [0, 1], '--', label='perfecta')
plt.xlabel('probabilidad estimada'); plt.ylabel('fracción real')
plt.title('Calibración'); plt.legend(); plt.show()
print('Brier:', round(brier_score_loss(y_train, proba), 4))
prec, rec, umbrales = precision_recall_curve(y_train, proba)
f1s = 2 * prec * rec / (prec + rec + 1e-9)
mejor = int(np.nanargmax(f1s[:-1]))
print('umbral que maximiza F1:', round(float(umbrales[mejor]), 3),
      '| F1:', round(float(f1s[mejor]), 3))

### Paso 54. Errores por subgrupos

In [ ]:
errores = X_train.copy()
errores['y_real'] = y_train
errores['pred'] = pred
errores['error'] = (pred != y_train.to_numpy())
errores['edad_grupo'] = pd.cut(errores['Age'], [0, 16, 40, 100],
                               labels=['niño', 'adulto', 'mayor'])
for col in ['Sex', 'Pclass', 'edad_grupo']:
    print(col, errores.groupby(col, observed=True)['error'].agg(['mean', 'count']).round(3))
    print()

In [ ]:
# 10 errores de alta confianza: proba muy alta o muy baja con predicción equivocada.
conf = pd.Series(np.abs(proba - 0.5), index=errores.index)
malos = errores.loc[errores['error']].assign(proba=proba[np.asarray(errores['error'])])
malos = malos.assign(confianza=conf.loc[malos.index]).sort_values('confianza', ascending=False)
malos.head(10)

### Paso 55. Curva de aprendizaje

In [ ]:
from sklearn.model_selection import learning_curve

sizes, tr, va = learning_curve(pipe_finalista, X_train, y_train, cv=cv,
                               train_sizes=np.linspace(0.2, 1.0, 5),
                               scoring='roc_auc')
plt.plot(sizes, tr.mean(axis=1), 'o-', label='entrenamiento')
plt.fill_between(sizes, tr.mean(axis=1) - tr.std(axis=1), tr.mean(axis=1) + tr.std(axis=1), alpha=0.2)
plt.plot(sizes, va.mean(axis=1), 'o-', label='validación')
plt.fill_between(sizes, va.mean(axis=1) - va.std(axis=1), va.mean(axis=1) + va.std(axis=1), alpha=0.2)
plt.xlabel('tamaño de entrenamiento'); plt.ylabel('ROC-AUC')
plt.title('Curva de aprendizaje'); plt.legend(); plt.show()

## 13. Ajuste de hiperparámetros (Pasos 56-58)

In [ ]:
from sklearn.model_selection import GridSearchCV

grids = ml.get_param_grids()
ajustes = {}
for nombre in ['logreg', 'forest', 'hgb']:
    gs = GridSearchCV(ml.build_pipeline(ml.get_candidates()[nombre]), grids[nombre],
                      scoring=ml.PRIMARY_METRIC, cv=cv, n_jobs=-1, refit=True)
    gs.fit(X_train, y_train)
    ajustes[nombre] = gs
    print(nombre, '|', gs.best_params_, '|', round(gs.best_score_, 4))

### Paso 58. ¿Mejoró de verdad? (comparación pareada con los mismos pliegues)

In [ ]:
pipe_base = ml.build_pipeline(ml.get_candidates()['hgb'])
res_base = cross_validate(pipe_base, X_train, y_train, cv=cv, scoring=[ml.PRIMARY_METRIC])
pipe_tuned = ml.build_pipeline(ml.get_candidates()['hgb']).set_params(**ajustes['hgb'].best_params_)
res_tuned = cross_validate(pipe_tuned, X_train, y_train, cv=cv, scoring=[ml.PRIMARY_METRIC])
dif = res_tuned[f'test_{ml.PRIMARY_METRIC}'] - res_base[f'test_{ml.PRIMARY_METRIC}']
print('diferencia por pliegue:', np.round(dif, 3).tolist())
print('media:', round(dif.mean(), 4), '| desviación:', round(dif.std(), 4))
# Si la mejora no supera la variabilidad entre pliegues, el ajuste aporta poco.

## 14. Interpretación y ablación (Pasos 59-61)

In [ ]:
pipe_lr = ml.build_pipeline(ml.get_candidates()['logreg']).fit(X_train, y_train)
nombres = pipe_lr['prep'].get_feature_names_out()
coefs = pipe_lr['model'].coef_[0]
coeficientes = (pd.DataFrame({'feature': nombres, 'coef': coefs, 'odds_ratio': np.exp(coefs)})
                .reindex(pd.Series(np.abs(coefs), index=nombres)
                         .sort_values(ascending=False).index))
coeficientes.head(15)

In [ ]:
from sklearn.inspection import permutation_importance

X_tr2, X_val, y_tr2, y_val = ml.split(X_train, y_train, test_size=0.25)
pipe_imp = ml.build_pipeline(ml.get_candidates()[nombre_finalista]).fit(X_tr2, y_tr2)
r = permutation_importance(pipe_imp, X_val, y_val, scoring='roc_auc',
                           n_repeats=30, random_state=ml.settings.random_state)
importancias = pd.Series(r.importances_mean, index=X_train.columns).sort_values()
importancias.plot.barh(xerr=pd.Series(r.importances_std, index=X_train.columns)[importancias.index])
plt.title('Importancia por permutación (datos no vistos)')
plt.show()

### Paso 61. Ablación de grupos de variables

In [ ]:
configs = {
    'A base': (ml.NUM_FEATURES, ml.CAT_FEATURES),
    'B base + cabina': (ml.NUM_FEATURES + ml.CABIN_NUM, ml.CAT_FEATURES + ml.CABIN_CAT),
    'C sin TTitulo': (ml.NUM_FEATURES, [c for c in ml.CAT_FEATURES if c != 'TTitulo']),
    'D sin Sex': (ml.NUM_FEATURES, [c for c in ml.CAT_FEATURES if c != 'Sex']),
}
filas = []
for nombre, (num, cat) in configs.items():
    res = cross_validate(ml.build_pipeline(ml.get_candidates()[nombre_finalista], num, cat),
                         X_train, y_train, cv=cv, scoring=[ml.PRIMARY_METRIC])
    filas.append({'config': nombre,
                  'roc_auc_mean': round(res[f'test_{ml.PRIMARY_METRIC}'].mean(), 4),
                  'roc_auc_std': round(res[f'test_{ml.PRIMARY_METRIC}'].std(), 4)})
pd.DataFrame(filas)

## 15. Evaluación final: el test se abre UNA vez (Pasos 62-63)

In [ ]:
pipe_eval = (ml.build_pipeline(ml.get_candidates()[ml.FINAL_ESTIMATOR])
             .set_params(**ml.FINAL_PARAMS).fit(X_train, y_train))
proba_test = pipe_eval.predict_proba(X_test)[:, 1]
res_test = ml.evaluate_on_test(pipe_eval, X_test, y_test, ml.THRESHOLD)
lo, hi = ml.bootstrap_auc_ci(y_test, proba_test)
print(res_test)
print(f'IC 95% ROC-AUC: [{lo:.3f}, {hi:.3f}]')
# Con ~180 pasajeros de prueba el intervalo es ancho: no comparar modelos por décimas.

In [ ]:
# Comparación final: baselines vs. modelo elegido (CV en entrenamiento + prueba).
pd.DataFrame({
    'modelo': ['dummy (CV)', 'regla de género', 'modelo elegido (CV)', 'modelo elegido (prueba)'],
    'exactitud': [res_dummy['test_accuracy'].mean(), accuracy_score(y_train, pred_genero),
                  tabla.loc[tabla['model'] == ml.FINAL_ESTIMATOR, 'test_accuracy_mean'].iloc[0],
                  res_test['accuracy']],
    'roc_auc': [res_dummy['test_roc_auc'].mean(), np.nan,
                tabla.loc[tabla['model'] == ml.FINAL_ESTIMATOR, 'test_roc_auc_mean'].iloc[0],
                res_test['roc_auc']],
}).round(3)

## 16. Modelo guardado y prueba de humo (Paso 69)

El modelo definitivo se entrena y guarda con `make train` (flujo `flows/05_ml_models.run()`), con TODOS los datos etiquetados. Aquí se carga y se prueba sobre pasajeros sin etiqueta.

In [ ]:
pipe_guardado, meta = ml.load_model(MODELS / 'titanic_model_v1.joblib')
meta

In [ ]:
de = importlib.import_module('flows.01_data_engineering')

nuevos = pd.read_csv(BRONZE / BRONZE_UNLABELED_NAME)
# Los sin etiqueta no tienen Survived; un 0 provisional SOLO permite que build_silver
# corra (esa columna no genera features del modelo). Las demás transformaciones son
# las MISMAS de Silver.
nuevos['Survived'] = 0
nuevos_silver = de.build_silver(nuevos)
X_nuevos = nuevos_silver[ml.CANDIDATE_COLUMNS].copy()
X_nuevos['Age'] = nuevos_silver['AgeOriginal']
predicciones = ml.predict(pipe_guardado, X_nuevos)
print('¿algún NaN?:', predicciones.isna().any().any())
predicciones['proba_survive'].describe()

## Limitaciones (Paso 63)

- Un solo suceso histórico; 891 pasajeros y ~180 de prueba (intervalos anchos).
- Edad y cabina con muchos faltantes; la imputación introduce sesgo documentado.
- `Sex` y `Pclass` dominaron decisiones de 1912: predecir lo que ocurrió no implica
  que deba ocurrir ni que esas variables deban usarse para decidir hoy.
- Detalle completo en `reports/problema_ml.md` y `reports/model_card.md`.